# Pending audit exposure in trustworthy AI services
## A reproducible synthetic feasibility study — 17 October 2026
**Working article:** *When Audit Labels Arrive Late: A Safety–Availability Frontier for Trustworthy AI Services*.

This is an executed proof of concept, not a validated deployment method or completed manuscript. It uses a NumPy logistic classifier trained on simulated features, and known simulated labels hidden from admission policies until their scheduled audit. No public benchmark or real audit delay is claimed. Requires Python 3 and NumPy only. Run cells in order. Exports results under `pending_audit_results/` in the current directory.

### Hypotheses fixed before this execution
H1: Ignoring pending exposure permits violations of a cumulative error envelope under harmful drift, particularly when errors are audited later.
H2: Worst-case pending reservation preserves that envelope for every stream, at a measurable coverage cost as feedback slows or disappears.
H3: Against a fixed pending cap with the same initial worst-case allowance, a dynamic reserve can increase coverage on benign streams by reusing verified correct outcomes. This is an empirical hypothesis, not a dominance theorem.

The primary estimand is paired coverage difference across independent simulated seeds. Safety is a pathwise inequality, not a p-value. No claims of novelty of elementary budget accounting or of universal coverage superiority are made.

In [1]:
import numpy as np
import json, csv, pathlib, platform, hashlib, heapq, itertools
from collections import defaultdict
ROOT = pathlib.Path('pending_audit_results'); ROOT.mkdir(exist_ok=True)
CONFIG = dict(n=3000, change=1500, train_n=1800, seeds=30, epsilon=0.10,
              initial_allowance=20, confidence_threshold=0.80,
              delays=[1, 20, 100, 400], scenarios=['benign', 'shift_independent', 'shift_error_delay', 'shift_missing_errors'])
POLICIES = ['static', 'confirmed_only', 'pending_cap', 'reserve', 'oracle_immediate']
print(json.dumps(CONFIG, indent=2))
print('Python:',platform.python_version(),'NumPy:',np.__version__)


{
  "n": 3000,
  "change": 1500,
  "train_n": 1800,
  "seeds": 30,
  "epsilon": 0.1,
  "initial_allowance": 20,
  "confidence_threshold": 0.8,
  "delays": [
    1,
    20,
    100,
    400
  ],
  "scenarios": [
    "benign",
    "shift_independent",
    "shift_error_delay",
    "shift_missing_errors"
  ]
}
Python: 3.9.6 NumPy: 2.0.2


## Exact statement and boundary
Let N be accepted releases, E confirmed errors, and P unresolved accepted outcomes. All losses are binary and audits are truthful. The gate admits only when

\[ E+P+1\leq\epsilon(N+1)+B. \]

An unresolved release costs one worst-case error unit; N increases by one. A correct audit removes one pending unit and an incorrect audit transfers one pending unit to E. Consequently E+P never increases on audit, and induction gives actual errors L ≤ E+P ≤ εN+B for every prefix, under arbitrary scores, outcomes, delays, and censoring. No exchangeability assumption is used. The loss must be bounded and each accepted release must remain in the ledger until a truthful audit; no timeout may silently release credit.

This is an **additive error envelope**: L/N ≤ ε+B/N, not a pure ε selective-risk guarantee. B is an explicitly tolerated startup allowance; B=0 and ε<1 permit no initial release. With no audit completion, N≤floor(B/(1−ε)). The bound is necessary as well as sufficient for a next release when every unresolved outcome may independently be wrong, and no extra information restricts them. This elementary result alone is not a substantial novelty claim.

Deferred requests are withheld. They do not earn credit and do not return automatic correct labels. Their workload is reported separately. The oracle sees immediate truthful outcomes only *after* its own release and is a diagnostic, not a deployable baseline. A fixed pending cap is an engineering baseline without a cumulative safety guarantee.

In [2]:
def sigmoid(z):
    return 1/(1+np.exp(-np.clip(z,-40,40)))

def make_stream(seed, scenario):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(CONFIG['train_n'],3))
    y = (rng.random(len(X)) < sigmoid(3*X[:,0]-2*X[:,1]+0.5*X[:,2])).astype(int)
    w = np.zeros(3)
    for _ in range(200):
        w -= 0.15*(np.sum(X*(sigmoid(np.sum(X*w,axis=1))-y)[:,None],axis=0)/len(y)+0.002*w)
    Z = rng.normal(size=(CONFIG['n'],3))
    truth = (rng.random(len(Z)) < sigmoid(3*Z[:,0]-2*Z[:,1]+0.5*Z[:,2])).astype(int)
    flip_draw = rng.random(len(Z))
    if scenario != 'benign':
        idx = np.arange(len(Z)) >= CONFIG['change']
        truth = np.where(idx & (flip_draw<0.55),1-truth,truth)
    assert np.isfinite(w).all()
    prob = sigmoid(np.sum(Z*w,axis=1))
    assert np.isfinite(prob).all()
    score = np.maximum(prob,1-prob)
    loss = ((prob>=0.5)!=truth).astype(int)
    return score, loss

def simulate(score, loss, base_delay, scenario, policy, keep_trace=False):
    eps, B = CONFIG['epsilon'], CONFIG['initial_allowance']
    N=E=P=L=0
    pending=[]; trace=[]; max_excess=0.; max_pending=0
    postN=postL=0
    for t,(conf,hidden_loss) in enumerate(zip(score,loss)):
        while pending and pending[0][0] <= t:
            _,audit_loss = heapq.heappop(pending)
            P-=1; E+=audit_loss
        eligible = conf >= CONFIG['confidence_threshold']
        if policy in ['reserve','oracle_immediate']:
            eligible = eligible and E+P+1 <= eps*(N+1)+B+1e-10
        elif policy == 'confirmed_only':
            eligible = eligible and E+1 <= eps*(N+1)+B+1e-10
        elif policy == 'pending_cap':
            eligible = eligible and P < int(B/(1-eps))
        if eligible:
            N+=1; P+=1; L+=int(hidden_loss)
            if t >= CONFIG['change']: postN+=1; postL+=int(hidden_loss)
            delay=base_delay
            if scenario=='shift_error_delay' and hidden_loss: delay*=10
            if scenario=='shift_missing_errors' and hidden_loss: delay=10**12
            if policy=='oracle_immediate':
                P-=1; E+=int(hidden_loss)
            else:
                heapq.heappush(pending,(t+delay,int(hidden_loss)))
        excess=max(0.,L-eps*N-B)
        max_excess=max(max_excess,excess); max_pending=max(max_pending,P)
        if policy in ['reserve','oracle_immediate']:
            assert L <= E+P and E+P <= eps*N+B+1e-8
        if keep_trace: trace.append([t,N,L,E,P,eps*N+B])
    return dict(coverage=N/len(loss), accepted=N, selective_risk=L/N if N else None,
                post_coverage=postN/(len(loss)-CONFIG['change']),
                post_risk=postL/postN if postN else None,
                max_excess=max_excess, violation=int(max_excess>1e-8),
                max_pending=max_pending, pending_end=P, deferred=len(loss)-N), trace

# Boundary checks, including all-wrong and no-completion outcomes.
for loss in [np.zeros(100,dtype=int), np.ones(100,dtype=int)]:
    r,_=simulate(np.ones(100),loss,10**9,'benign','reserve')
    assert r['accepted']==int(CONFIG['initial_allowance']/(1-CONFIG['epsilon']))
print('Boundary checks passed: no completed audits permit exactly 22 releases at B=20, epsilon=0.1.')


Boundary checks passed: no completed audits permit exactly 22 releases at B=20, epsilon=0.1.


## Paired factorial experiment
All policies receive identical scores, latent losses and potential audit delays for a seed/scenario. Policy inputs exclude latent losses before audit. Error-dependent and missingness mechanisms are generated by the environment using latent outcomes; the gate does not observe those outcomes. Matched random seeds across scenarios isolate label reversal while keeping classifier and score streams fixed. Seeds are simulation replications, not independent real datasets. The 95% bootstrap intervals below describe Monte Carlo variation in this simulator only. No hyperparameter tuning on these streams is performed.

In [3]:
rows=[]
for scenario in CONFIG['scenarios']:
    for seed in range(CONFIG['seeds']):
        score,loss=make_stream(seed,scenario)
        for delay in CONFIG['delays']:
            for policy in POLICIES:
                r,_=simulate(score,loss,delay,scenario,policy)
                rows.append(dict(scenario=scenario,seed=seed,delay=delay,policy=policy,**r))
with (ROOT/'runs.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
summary=[]
for scenario,delay,policy in itertools.product(CONFIG['scenarios'],CONFIG['delays'],POLICIES):
    rr=[r for r in rows if (r['scenario'],r['delay'],r['policy'])==(scenario,delay,policy)]
    summary.append(dict(scenario=scenario,delay=delay,policy=policy,
                        coverage_mean=float(np.mean([r['coverage'] for r in rr])),
                        coverage_median=float(np.median([r['coverage'] for r in rr])),
                        coverage_sd=float(np.std([r['coverage'] for r in rr],ddof=1)),
                        violation_runs=sum(r['violation'] for r in rr),
                        worst_excess=max(r['max_excess'] for r in rr),
                        selective_risk_mean=float(np.mean([r['selective_risk'] for r in rr if r['selective_risk'] is not None]))))
with (ROOT/'summary.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(summary[0]));writer.writeheader();writer.writerows(summary)
assert not any(r['violation'] for r in rows if r['policy']=='reserve')
print('Completed',len(rows),'policy runs;',len(rows)*CONFIG['n'],'request decisions.')
print('Scenario                  Delay Policy             Coverage  Violations/30  Worst excess')
for r in summary:
    if r['delay']==100:
        print(f"{r['scenario']:25s} {r['delay']:4d} {r['policy']:18s} {r['coverage_mean']:7.3f} {r['violation_runs']:8d}/30 {r['worst_excess']:12.2f}")


Completed 2400 policy runs; 7200000 request decisions.
Scenario                  Delay Policy             Coverage  Violations/30  Worst excess
benign                     100 static               0.582        0/30         0.00
benign                     100 confirmed_only       0.582        0/30         0.00
benign                     100 pending_cap          0.220        0/30         0.00
benign                     100 reserve              0.467        0/30         0.00
benign                     100 oracle_immediate     0.582        0/30         0.00
shift_independent          100 static               0.582       30/30       353.30
shift_independent          100 confirmed_only       0.370       30/30        40.10
shift_independent          100 pending_cap          0.220       30/30       123.00
shift_independent          100 reserve              0.219        0/30         0.00
shift_independent          100 oracle_immediate     0.347        0/30         0.00
shift_error_delay         

In [4]:
# Paired differences: reserve minus fixed pending cap; four descriptive comparisons.
rng=np.random.default_rng(934)
paired=[]
for delay in CONFIG['delays']:
    a=np.array([r['coverage'] for r in rows if r['scenario']=='benign' and r['delay']==delay and r['policy']=='reserve'])
    b=np.array([r['coverage'] for r in rows if r['scenario']=='benign' and r['delay']==delay and r['policy']=='pending_cap'])
    diff=a-b
    boot=diff[rng.integers(0,len(diff),(10000,len(diff)))].mean(axis=1)
    lo,hi=np.quantile(boot,[.025,.975])
    paired.append(dict(delay=delay,mean_difference=float(diff.mean()),median_difference=float(np.median(diff)),lo=float(lo),hi=float(hi)))
print('Benign coverage: reserve minus pending cap; paired bootstrap 95% intervals (descriptive, unadjusted)')
for p in paired: print(p)
(ROOT/'paired_intervals.json').write_text(json.dumps(paired,indent=2))
# Save one transparent replay, using evaluator-only true losses in separate columns.
score,loss=make_stream(0,'shift_error_delay')
for policy in POLICIES:
    _,trace=simulate(score,loss,100,'shift_error_delay',policy,True)
    with (ROOT/f'trace_{policy}.csv').open('w',newline='') as f:
        writer=csv.writer(f);writer.writerow(['t','accepted','true_errors_evaluator_only','confirmed_errors','pending','envelope']);writer.writerows(trace)
manifest=dict(config=CONFIG,python=platform.python_version(),numpy=np.__version__,
              data_type='synthetic logistic classification; no external dataset',
              stream_seed0_sha256=hashlib.sha256(score.tobytes()+loss.tobytes()).hexdigest(),
              guarantee='cumulative actual errors <= epsilon*accepted + B; not pure epsilon risk',
              inference='bootstrap intervals quantify seed variability only')
(ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Results saved to',ROOT.resolve())


Benign coverage: reserve minus pending cap; paired bootstrap 95% intervals (descriptive, unadjusted)
{'delay': 1, 'mean_difference': 0.0, 'median_difference': 0.0, 'lo': 0.0, 'hi': 0.0}
{'delay': 20, 'mean_difference': 0.0, 'median_difference': 0.0, 'lo': 0.0, 'hi': 0.0}
{'delay': 100, 'mean_difference': 0.24729999999999996, 'median_difference': 0.24833333333333332, 'lo': 0.24123333333333333, 'hi': 0.25286694444444446}
{'delay': 400, 'mean_difference': 0.01558888888888889, 'median_difference': 0.01633333333333334, 'lo': 0.01407777777777778, 'hi': 0.016977777777777776}
Results saved to /Users/habibhamam/Documents/Codex/2026-10-16/https-www-mdpi-com-journal-applsci/outputs/pending_audit_results


## Interpretation and falsification
The reserve policy must satisfy the envelope by construction; observing zero violations checks the implementation, not empirical superiority. Its value must be assessed by usable coverage relative to comparably safe alternatives. The pending cap and confirmed-only ablation have different guarantees and cannot establish state-of-the-art performance. Fast-audit oracle results show the attainable value of feedback, not an online benchmark win.

A high final risk can still satisfy the additive envelope when few requests are accepted. Report B/N, all prefix excesses, deferred load, and remaining unresolved cases. Cumulative safety can also hide a recent unsafe period; sliding-window or group safety would require separate ledgers and proofs. Permanent missingness may cause lasting shutdown. This is an honest limit, not successful service provision.

**Publication gate:** do not claim a novel algorithm or a completed empirical paper from this notebook. Require full-text comparison with conservative bandits, delayed/censored-feedback methods, sequential risk monitoring, and recent audit-capacity/availability work. Then evaluate frozen predictors on at least two real public tasks with chronological or declared controlled shifts; simulated audit times must remain labeled simulated. Add a properly implemented published anytime-valid monitor and a calibrated conservative baseline, tune on a separate planning split, and compare coverage at the same error-envelope requirement. Audit prioritization is evaluated in the exploratory extension below; it is not a confirmed improvement or a globally optimal policy.

## Literature and provenance
- Podkopaev and Ramdas (2021), *Tracking the risk of a deployed model and detecting harmful distribution shifts*, arXiv:2110.06177. Retrieved through Consensus and fetched before use. https://doi.org/10.48550/arXiv.2110.06177
- Wang, Wang, Johansson and Hirche (2024), *Risk-averse learning with delayed feedback*, arXiv:2409.16866. Retrieved and fetched through PaperGuide. https://doi.org/10.48550/arXiv.2409.16866
- Priye et al. (2026), *Available Guardrails: Certifying Selective Prediction across ML Systems*, arXiv:2609.22048. Consensus fetched record; recent preprint, full-text novelty comparison remains necessary. https://doi.org/10.48550/arXiv.2609.22048
- Jie Deng (2026), *When Can Fraud Operations Authorize Automation? A Decision-Support Framework for Fresh Audit Evidence and Review Workload*, arXiv:2608.08577. Fetched through Consensus and cross-checked against the primary arXiv full text; complete novelty comparison before using in the article. https://doi.org/10.48550/arXiv.2608.08577

Discovery records are not proof that the literature search is exhaustive. No synthetic result is evidence about a real human-review service.

## Exploratory extension: which audit should be completed next?
This extension was added after the initial pilot and is **exploratory**, not a preregistered confirmation. A single non-preemptive auditor takes five request ticks per audit (50 ticks for errors in the outcome-dependent scenario). Compare FIFO, highest-confidence, lowest-confidence, and an age-capped highest-confidence rule. The model score is not recalibrated after drift. All choices depend on observed confidence/age only. Releasing reserve requires a completed correct audit; finding an error transfers pending exposure to confirmed exposure and releases no credit. Therefore error discovery and immediate reserve recovery are different objectives. Neither objective alone establishes long-run optimality. No arbitrary citation-based percentage target is used.

In [5]:
def scheduled(score, loss, mode, scenario):
    eps,B=CONFIG['epsilon'],CONFIG['initial_allowance']
    N=E=L=0; waiting=[]; active=None; completed=0; max_age=0; trace=[]
    for t,conf in enumerate(score):
        if active is not None and active[0]<=t:
            E+=active[2];completed+=1;active=None
        P=len(waiting)+int(active is not None)
        if conf>=CONFIG['confidence_threshold'] and E+P+1<=eps*(N+1)+B+1e-10:
            N+=1;L+=int(loss[t]);waiting.append((t,float(conf),int(loss[t])))
        if active is None and waiting:
            if mode=='fifo': k=0
            elif mode=='high_confidence': k=max(range(len(waiting)),key=lambda j:waiting[j][1])
            elif mode=='uncertainty': k=min(range(len(waiting)),key=lambda j:waiting[j][1])
            else:
                k=0 if t-waiting[0][0]>=200 else max(range(len(waiting)),key=lambda j:waiting[j][1])
            origin,score_i,hidden=waiting.pop(k)
            duration=50 if scenario=='shift_error_delay' and hidden else 5
            active=(t+duration,origin,hidden)
        P=len(waiting)+int(active is not None)
        assert L<=E+P and E+P<=eps*N+B+1e-8
        if waiting:max_age=max(max_age,t-waiting[0][0])
        if active is not None:max_age=max(max_age,t-active[1])
    return dict(coverage=N/len(score),accepted=N,true_errors=L,completed_audits=completed,
                pending_end=P,max_pending_age=max_age,confirmed_errors=E)
schedule_rows=[]
for scenario in ['benign','shift_independent','shift_error_delay']:
    for seed in range(CONFIG['seeds']):
        scores,losses=make_stream(seed,scenario)
        for mode in ['fifo','high_confidence','uncertainty','age_capped']:
            schedule_rows.append(dict(scenario=scenario,seed=seed,scheduler=mode,**scheduled(scores,losses,mode,scenario)))
with (ROOT/'scheduler_runs.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(schedule_rows[0]));writer.writeheader();writer.writerows(schedule_rows)
print('Exploratory single-auditor results:')
for scenario in ['benign','shift_independent','shift_error_delay']:
    for mode in ['fifo','high_confidence','uncertainty','age_capped']:
        rr=[r for r in schedule_rows if r['scenario']==scenario and r['scheduler']==mode]
        print(scenario, mode, 'coverage=',round(float(np.mean([r['coverage'] for r in rr])),4),
              'max age=',max(r['max_pending_age'] for r in rr))


Exploratory single-auditor results:
benign fifo coverage= 0.2215 max age= 358
benign high_confidence coverage= 0.2235 max age= 2997
benign uncertainty coverage= 0.2209 max age= 2997
benign age_capped coverage= 0.2215 max age= 358
shift_independent fifo coverage= 0.1454 max age= 269
shift_independent high_confidence coverage= 0.1462 max age= 2257
shift_independent uncertainty coverage= 0.1455 max age= 2272
shift_independent age_capped coverage= 0.1453 max age= 269
shift_error_delay fifo coverage= 0.1105 max age= 1343
shift_error_delay high_confidence coverage= 0.1117 max age= 2996
shift_error_delay uncertainty coverage= 0.105 max age= 2997
shift_error_delay age_capped coverage= 0.1096 max age= 1319
